In [1]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
import openai
from dotenv import load_dotenv
import os
import shutil


In [2]:
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")
CHROMA_PATH = "./.chroma"
SOURCE_PATH = "data"

In [3]:
def main():
    generate_data_store()

# Markdown File Loading

In [4]:
def generate_data_store():
    documents = load_documents()
    texts = split_text(documents)
    save_to_chroma(texts)

In [5]:
def load_documents():
    loader = DirectoryLoader(
        SOURCE_PATH, glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},)
    documents = loader.load()
    return documents

In [6]:
def split_text(documents: list[Document]):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=100,
        length_function=len,
        add_start_index=True,
    )
    texts = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} into {len(texts)} chunks.")
    document = texts[10]
    print(document.page_content)
    print(document.metadata)
    return texts

In [7]:
def save_to_chroma(texts: list[Document]):
    if os.path.exists(CHROMA_PATH):
        shutil.rmtree(CHROMA_PATH, ignore_errors=True)
    
    db = Chroma.from_documents(
        texts,
        OpenAIEmbeddings(),
        persist_directory=CHROMA_PATH
    )
    
    print(f"saved {len(texts)} to chroma database {CHROMA_PATH}.")

In [8]:
if __name__ == "__main__":
    main()

Split into 1 into 812 chunks.
So she was considering in her own mind (as well as she could, for the
hot day made her feel very sleepy and stupid), whether the pleasure of
making a daisy-chain would be worth the trouble of getting up and
picking the daisies, when suddenly a White Rabbit with pink eyes ran
close by her.
{'source': 'data/alice_in_wonderland.md', 'start_index': 1661}
saved 812 to chroma database ./.chroma.


# PDF Loaders

# WebLoader